<a href="https://colab.research.google.com/github/Adrianoglima22/Agno-Examples/blob/main/CanarIA-tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install -q agno openai tavily-python wikipedia

In [ ]:
import os
from google.colab import userdata
from agno.tools.wikipedia import WikipediaTools
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

# 1 - Recriando o treinador sem ferramenta

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

modelo_openai = OpenAIChat(id="gpt-5.4-nano")

treinador = Agent(
    name="treinador",
    description="Assistente do CanarIA sobre a seleção Brasileira masculina de futebol.",
    model=modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do CanarIA dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",
    ],
    markdown=True,
)

In [ ]:
treinador.print_response(
    "Quantos gols o Vini Jr tem pela seleção?",
    stream=True,
)

# 2 - Utilizando a tool (Tavily)

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

modelo_openai = OpenAIChat(id="gpt-5.4-mini")

treinador = Agent(
    name="treinador",
    description="Assistente CanarIA sobre a seleção Brasileira masculina de futebol.",
    model=modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do CanarIA dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",
    ],
    tools=[TavilyTools(), WikipediaTools()],  # = duas tools
    markdown=True,
)

A mesma pergunta, agora com tool

In [ ]:
treinador.print_response(
    "Quantos gols o Vini Jr tem pela seleção?",
    stream=True,
)

Observando o agente decidir não utilizar a tool

In [ ]:
treinador.print_response(
    "Qual a vantagem tática de jogar com dois volantes contra um adversário que pressiona alto?",
    stream=True,
)

# 3 - Refinando as instruções de uso de ferramentas

In [ ]:
treinador = Agent(
    name="treinador",
    description="Assistente do CanarIA sobre a seleção Brasileira masculina de futebol.",
    model=modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do CanarIA dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",

        """# Nova instruction: guia o uso da tool
        "Use a busca web (Tavily) sempre que a pergunta envolver: "
        "eventos recentes (últimos jogos, convocações, lesões),"
        "estatísticas verificáveis (número de gols, recordes, classificações)"
        "ou forma atual de jogadores."
        "Para perguntas conceituais (táticas, regras, princípios de jogo)"
        "ou históricas consolidadas (copas antigas, técnicos do passado),"
        "responda direto sem busca."
        """,

    ],
    tools=[TavilyTools(), WikipediaTools()],  # = duas tools
    markdown=True,
)

In [ ]:
# Pergunta com fato verificável recente + deve usar a tool
treinador.print_response(
    "Como o Vinícius Júnior tem se saído nos últimos jogos pelo Real Madrid?",
    stream=True,
)

In [ ]:
# Pergunta tática conceitual = não deve usar a tool
treinador.print_response(
    "Em que situações um treinador prefere marcação por zona em vez de marcação individual?",
    stream=True,
)

# Testando a pergunta híbrida sem política



In [ ]:
treinador.print_response(
    "Como Telê Santana montava o ataque da Seleção de 82, e quem hoje teria perfil parecido na última convocação?",
    stream=True,
)

# Definindo a política de fontes

In [ ]:
treinador = Agent(
    name="treinador",
    description="Assistente do CanarIA sobre a seleção Brasileira masculina de futebol.",
    model=modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do CanarIA dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",

        # Instruction central: politica de uso de tools
        """POLÍTICA DE FONTES - siga rigorosamente":
        "- Para EVENTOS RECENTES (últimos jogos, convocações, lesões, forma atual):"
        "use Tavily (busca web)."
        "- Para FATOS HISTÓRICOS CONSOLIDADOS (copas antigas, biografias, técnicos do passado, "
        "regulamentos): use wikipedia - é mais estruturada e citável."
        "- Para PERGUNTAS CONCEITUAIS (táticas, regras, análise interpretativa):"
        "responda direto sem tool."
        "- Quando a pergunta tiver MÚLTIPLAS NECESSIDADES, combine fontes:"
        "Wikipedia para a parte histórica, Tavily para a parte recente, "
        "e sua análise para a parte interpretativa."
        """,

    ],
    tools=[TavilyTools(), WikipediaTools()],  # = duas tools
    markdown=True,
)

# Validando a versão sem tools

In [ ]:
# agente sem tools - só conhecimento interno do modelo
treinador_sem_tool = Agent(
    name="Treinador",
    description="Assistente do CanarIA sobre a Seleção Brasileira masculina de futebol.",
    model=modelo_openai,
    instructions="""Você é o Treinador, assistente do CanarIA dedicado à Seleção Brasileira masculina.
"Responda em português do Brasil, com tom profissional e analítico.",
"Quando não tiver certeza de um dado, diga claramente."
""",
    markdown=True,
)

treinador_sem_tool.print_response(
    "Como Telê Santana montava o ataque da Seleção de 82, e quem hoje teria perfil parecido na última convocação?",
    stream=True,
)

# Avaliando o comportamento com ferramentas sem política

In [ ]:
treinador_sem_politica = Agent(
    name="Treinador",
    description="Assistente do CanarIA sobre a Seleção Brasileira masculina de futebol.",
    model=modelo_openai,
    instructions="""Você é o Treinador, assistente do CanarIA dedicado à Seleção Brasileira masculina.
"Responda em português do Brasil, com tom profissional e analítico.",
"Quando não tiver certeza de um dado, diga claramente."
""",
    tools=[TavilyTools(), WikipediaTools()],
    markdown=True,
)

# Aplicando a política e reduzindo chamadas desnecessárias

In [ ]:
treinador.print_response(
    "Como Telê Santana montava o ataque da Seleção de 82, e quem hoje teria perfil parecido na última convocação?",
    stream=True,
)

# Comparando versões e checando uso consciente de ferramentas

In [ ]:
treinador.print_response(
    "Em que cenários um esquema com 3 zagueiros é mais eficaz que um com 4?",
    stream=True,
)